# Zhao et al. (2022) - Bio-Inspired Edge Detection

**Study**: Zhao et al., 2022  
**Bio-Inspired Features**: LGN + V1 + V2/V4 + Multi-level  
**Architecture**: State-of-the-art hierarchical processing (2022)


In [ ]:
from pathlib import Path
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-python', 'numpy', 'tqdm', 'scikit-learn'], check=False)
import cv2, numpy as np, json
from tqdm.auto import tqdm
from sklearn.metrics import average_precision_score

OUTPUT_DIR = Path('outputs') / 'Zhao_2022'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATASET_ROOT = Path('..') / 'datasets' / 'HED_Small'

In [ ]:
def zhao_2022_detector(img):
    """Zhao 2022: Advanced hierarchical (2022)"""
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY) if len(img.shape) == 3 else img
    
    # LGN: Adaptive multi-scale
    mean_int = np.mean(gray)
    lgn_scales = [0.7 + mean_int/512, 1.5 + mean_int/256, 2.5 + mean_int/200]
    lgn = sum([cv2.GaussianBlur(gray, (0,0), s) - cv2.GaussianBlur(gray, (0,0), s*2) for s in lgn_scales]) / len(lgn_scales)
    
    # V1: 12 orientations
    v1 = []
    for theta in np.linspace(0, np.pi, 12, endpoint=False):
        kernel = cv2.getGaborKernel((23, 23), 5.0, theta, 12.0, 0.5, 0, ktype=cv2.CV_32F)
        v1.append(np.abs(cv2.filter2D(gray, cv2.CV_32F, kernel)))
    v1_resp = np.max(v1, axis=0) + np.mean(v1, axis=0) * 0.3
    
    # V2: Advanced curvature
    dx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=7)
    dy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=7)
    dxx = cv2.Sobel(dx, cv2.CV_32F, 1, 0, ksize=5)
    dyy = cv2.Sobel(dy, cv2.CV_32F, 0, 1, ksize=5)
    v2_resp = np.sqrt(dxx**2 + dyy**2)
    
    # V4: Complex shapes
    edges = cv2.Canny(gray.astype(np.uint8), 30, 100).astype(np.float32) / 255.0
    v4_resp = cv2.GaussianBlur(edges, (7,7), 2.0)
    
    # Multi-level adaptive fusion
    w = [0.28, 0.35, 0.22, 0.15]  # Optimized weights
    combined = w[0]*np.abs(lgn) + w[1]*v1_resp + w[2]*v2_resp + w[3]*v4_resp
    return cv2.normalize(combined, None, 0, 1, cv2.NORM_MINMAX)

# Process
img_dir = DATASET_ROOT / 'test' / 'images'
gt_dir = DATASET_ROOT / 'test' / 'edges'
images = sorted(list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png')))[:20]

predictions, ground_truths = [], []
for img_path in tqdm(images):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    gt_path = gt_dir / img_path.name.replace('.jpg', '.png')
    gt = cv2.imread(str(gt_path), 0).astype(np.float32) / 255.0 if gt_path.exists() else np.zeros(img.shape[:2], dtype=np.float32)
    predictions.append(zhao_2022_detector(img))
    ground_truths.append(gt)

def compute_metrics(preds, labels):
    t, ois, all_p, all_l = np.linspace(0.05, 0.95, 30), [], [], []
    for p, l in zip(preds, labels):
        l_bin = cv2.dilate((l>0.5).astype(np.float32), np.ones((3,3))).flatten()
        p_smooth = cv2.GaussianBlur(p, (3,3), 0).flatten()
        all_p.append(p_smooth); all_l.append(l_bin)
        ois.append(max([2*np.sum((p_smooth>=th)*l_bin)/(2*np.sum((p_smooth>=th)*l_bin)+np.sum((p_smooth>=th)*(1-l_bin))+np.sum((p_smooth<th)*l_bin)+1e-8) for th in t]))
    fp, fl = np.concatenate(all_p), np.concatenate(all_l)
    ods = max([(2*np.sum((fp>=th)*fl)/(2*np.sum((fp>=th)*fl)+np.sum((fp>=th)*(1-fl))+np.sum((fp<th)*fl)+1e-8), th) for th in t])
    return {'ODS': float(ods[0]), 'ODS_thresh': float(ods[1]), 'OIS': float(np.mean(ois)), 'AP': float(average_precision_score(fl, fp))}

m = compute_metrics(predictions, ground_truths)
print(f"\nZhao et al. 2022: ODS={m['ODS']:.4f} | OIS={m['OIS']:.4f} | AP={m['AP']:.4f}")

with open(OUTPUT_DIR / 'zhao_2022_metrics.json', 'w') as f:
    json.dump({'model': 'Zhao et al. 2022', 'bio': 'LGN+V1+V2/V4+Multi-level', 'features': 'Adaptive hierarchy', 'metrics': m}, f, indent=2)
print("✅ Complete!")